In [ ]:
# -------------------------------------------
# ✅ Install dependencies (if not installed)
# -------------------------------------------

# You can uncomment these in Colab or local terminal if needed
!pip install nltk
!pip install keras       # or use tensorflow.keras instead
!pip install tensorflow

# -------------------------------------------
# ✅ Imports
# -------------------------------------------

import pandas as pd
import numpy as np
import nltk
from string import punctuation
from nltk.tokenize import RegexpTokenizer, WordPunctTokenizer, WhitespaceTokenizer
from nltk.corpus import stopwords

# For Keras Tokenizer
# from tensorflow.keras.preprocessing.text import Tokenizer
# OR:
import keras
from keras.preprocessing.text import Tokenizer

# Download stopwords if not already downloaded
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# -------------------------------------------
# ✅ Prepare Toy Data
# -------------------------------------------

sentence_list = [
    "First, we consider the initial spreadsheet model 123@.",
    "Then we consider another model @247, not the initial one.",
    "First, we consider the initial spreadsheet model and then we consider another model. So 2."
]

sentences_df = pd.DataFrame(sentence_list, columns=["sentence"])
pd.set_option('max_colwidth', 400)

sample_text = sentences_df.sentence.str.cat(sep=' ')

# -------------------------------------------
# ✅ NLTK Tokenizers
# -------------------------------------------

# RegexTokenizer using word pattern
TOKEN_PATTERN = r'\w+'
regex_wt = RegexpTokenizer(pattern=TOKEN_PATTERN, gaps=False)
tokens_word = regex_wt.tokenize(sample_text)

# RegexTokenizer using gap (whitespace)
GAP_PATTERN = r'\s+'
regex_gap = RegexpTokenizer(pattern=GAP_PATTERN, gaps=True)
tokens_gap = regex_gap.tokenize(sample_text)

# Get positions (spans) of tokens
word_indices = list(regex_gap.span_tokenize(sample_text))
token_spans = [sample_text[start:end] for start, end in word_indices]

# WordPunctTokenizer
wordpunkt_wt = WordPunctTokenizer()
tokens_wordpunkt = wordpunkt_wt.tokenize(sample_text)

# WhitespaceTokenizer
whitespace_wt = WhitespaceTokenizer()
tokens_whitespace = whitespace_wt.tokenize(sample_text)

# -------------------------------------------
# ✅ Text Cleaning
# -------------------------------------------

# Download stopwords if not already downloaded
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    tokens = text.split()
    table = str.maketrans('', '', punctuation)
    tokens = [word.translate(table) for word in tokens]
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word.lower() for word in tokens]
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [word for word in tokens if len(word) > 1]
    return ' '.join(tokens)

# Apply to full sample_text
print(clean_text(sample_text))

# Clean each sentence in DataFrame
sentences_df["sentence"] = sentences_df["sentence"].apply(clean_text)

# -------------------------------------------
# ✅ Keras Tokenizer
# -------------------------------------------

sentence_list_cleaned = list(sentences_df["sentence"])

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentence_list_cleaned) # 扫描文本，统计词频，编号
word_index = tokenizer.word_index

# View result
print(word_index)


# Text Preprocessing - Word Tokenization
> 文本预处理中的“词元化”（Tokenization）

### What is Tokenization?
the process of splitting text into smaller, meaningful components called __tokens__ — usually words or symbols
> 把长文本（sentence/document）切割成更小的部分 —— “tokens”（通常是词或符号）
### Why Tokenization?
It’s the first step in natural language processing (NLP) — crucial for preparing data for models.
## Different tokenizers
> NLTK 提供很多 tokenizer，这节课主要介绍了两种方式用 regex
### 1. NLTK's two Regex Tokenizers
- `nltk.tokenize.RegexpTokenizer` with `\w+`
    - `\w+`: Splits text into sequences of word characters (letters, numbers).
    - Example：`"First, we consider"` → `['First', 'we', 'consider']`
    ```bash
        TOKEN_PATTERN = r'\w+'
        regex_wt = nltk.RegexpTokenizer(pattern=TOKEN_PATTERN, gaps=False) # default is False, matches itself
        words = regex_wt.tokenize(sample_text) # output is a list
    ```
- `nltk.tokenize.RegexpTokenizer` with `\s+` + `gaps=True`
    - `\s+` with `gaps=True`: Splits on whitespace (spaces, tabs, newlines), keeping **punctuation** as part of tokens.
    - Example：`"First, we consider"` → `['First,', 'we', 'consider']`
    ```bash
        GAP_PATTERN = r'\s+'
        regex_wt = nltk.RegexpTokenizer(pattern=GAP_PATTERN, gaps=True)
        words = regex_wt.tokenize(sample_text)
    ```
- You can also extract the **positions** (start and end indices) of each token using `.span_tokenize(sampletext)`
    - .span_tokenize(sampletext)
        - Returns the start and end indices of each token in the original text.
        - Example: `['First', 'we', 'consider']` → `[(0, 5), (6, 8), (9, 17)]`
- `nltk.tokenize.WordPunctTokenizer`

### 2. NLTK's WordPunctTokenizer
- `nltk.tokenize.WordPunctTokenizer`: Splits text into words and punctuation.
    - Example: `"First, we consider"` → `['First', ',', 'we', 'consider']`
    ```bash
        word_punct_tokenizer = nltk.tokenize.WordPunctTokenizer()
        words = word_punct_tokenizer.tokenize(sample_text)
    ```
### 3. NLTK's WhitespaceTokenizer
- `nltk.tokenize.WhitespaceTokenizer`: Splits text based on whitespace.
    - Example: `"First, we consider"` → `['First,', 'we', 'consider']`
    ```bash
        whitespace_tokenizer = nltk.tokenize.WhitespaceTokenizer()
        words = whitespace_tokenizer.tokenize(sample_text)
    ```


# clean text


In [1]:
# Typical Example of text cleaning
def clean_text(text:str) -> str:
    from nltk.corpus import stopwords
    from string import punctuation
    stop_words = set(stopwords.words('english'))

    # Tokenize
    tokens = text.split()

    # Remove punctuation
    table = str.maketrans('', '', punctuation)
    tokens = [word.translate(table) for word in tokens]

    # Remove non-alphabetic
    tokens = [word for word in tokens if word.isalpha()]

    # Lowercase
    tokens = [word.lower() for word in tokens]

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Remove short words
    tokens = [word for word in tokens if len(word) > 1]

    return ' '.join(tokens)


# Keras Tokenizer
```text
raw text (list[sentences])\
  ↓ \
[NLTK] Tokenize → Apply Clean (remove stopwords, lowercase, etc.) (str -> str, apply -> list[cleaned_sentences])\
  ↓ \
cleaned text \
  ↓ \
[keras] fit tokenizer ➝ sequences ➝ use to model \
```
| 工具                                  | 来自哪里               | 主要用途                             | 返回类型                                             |
|---------------------------------------|------------------------|--------------------------------------|------------------------------------------------------|
| `nltk.tokenize.*`                     | NLTK（自然语言处理库） | 把文本“切成词” （text → words）      | `['this', 'is', 'a', 'sentence']`                    |
| `keras.preprocessing.text.Tokenizer`  | Keras（深度学习框架） | 把词“映射成数字” （word → integer） | `{'this': 1, 'is': 2, 'a': 3, ...}`                  |

> NLTK tokenizer 只负责把文本切成词，Keras tokenizer maps word to integer, the output dictionary is `word -> frequency ranking`


In [ ]:
# step 1: instantiate the tokenizer
tokenizer = Tokenizer()
# step 2: fit the tokenizer on the text
# the input needs to be a list of sentences
tokenizer.fit_on_texts(sentence_list_cleaned)
# the tokenizer will automatically 1. divide the text into tokens, 2. count the frequency of each token, 3. and assign a unique integer to each token
# step 3: get the word index
word_index = tokenizer.word_index